# CIE 2 – Activity 2
## Banknote Authentication Using Logistic Regression

**Student Roll Number:** AD3266

### Aim
Build a binary classification model using Logistic Regression to predict whether a banknote is genuine or forged.

### Dataset
This notebook uses the Banknote Authentication dataset from the UCI Machine Learning Repository.

The dataset contains four numerical input features:
- Variance
- Skewness
- Curtosis
- Entropy

**Target column:** `Class`
- `0` = Genuine
- `1` = Forged

The class-label interpretation follows the commonly used interpretation of this dataset.


## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

%matplotlib inline
sns.set_theme(style="whitegrid")


## 2. Load the dataset

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"

column_names = ["Variance", "Skewness", "Curtosis", "Entropy", "Class"]

df = pd.read_csv(url, header=None, names=column_names)

print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["Class"].value_counts().sort_index())


## 4. Exploratory Data Analysis (EDA)
Check summary statistics, class distribution, feature distributions and correlations.

In [ ]:
display(df.describe())

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Class")
plt.title("Distribution of Banknote Classes")
plt.xlabel("Class (0 = Genuine, 1 = Forged)")
plt.ylabel("Number of records")
plt.show()

df.drop(columns=["Class"]).hist(figsize=(12, 8), bins=20, edgecolor="black")
plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()


### 4.1 Correlation heatmap
The heatmap shows the linear correlation between the input features and the target class. Correlation does not by itself prove causation.

In [ ]:
plt.figure(figsize=(8, 5))
correlation = df.corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Banknote Features")
plt.tight_layout()
plt.show()


## 5. Separate input features and target
`X` contains the four numerical input features. `y` contains the binary class label.

In [ ]:
X = df[["Variance", "Skewness", "Curtosis", "Entropy"]]
y = df["Class"]

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)
print("Number of target classes:", y.nunique())


## 6. Split the data into training and testing sets
80% of the records are used for training and 20% for testing. Stratification preserves the class proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## 7. Scale the numerical features
Feature scaling puts the numerical features on comparable scales. The scaler is fitted only on the training data to avoid data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


## 8. Train the Logistic Regression model

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")


## 9. Make predictions

In [ ]:
y_pred = model.predict(X_test_scaled)

results = pd.DataFrame({
    "Actual Class": y_test.values,
    "Predicted Class": y_pred
})

display(results.head(10))


## 10. Evaluate model performance
Accuracy, precision, recall and F1-score are used to evaluate the classifier. Here, Class 1 (Forged) is treated as the positive class.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Genuine (0)", "Forged (1)"],
    zero_division=0
))


## 11. Confusion matrix
The confusion matrix compares actual classes with predicted classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Genuine (0)", "Forged (1)"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – Logistic Regression")
plt.show()


## 12. ROC curve and AUC score
The ROC curve shows the trade-off between the true positive rate and false positive rate at different classification thresholds. AUC summarizes the model's ability to distinguish the two classes.

In [ ]:
# Probability that each test record belongs to Class 1 (Forged)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {auc_score:.4f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Banknote Authentication")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("AUC Score:", round(auc_score, 4))


## 13. Visualize predicted probability distribution
This graph shows the model's predicted probability of a banknote being forged for genuine and forged test records.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    y_prob[y_test.to_numpy() == 0],
    label="Genuine Banknotes",
    kde=True,
    stat="density",
    alpha=0.4
)

sns.histplot(
    y_prob[y_test.to_numpy() == 1],
    label="Forged Banknotes",
    kde=True,
    stat="density",
    alpha=0.4
)

plt.xlabel("Predicted Probability of Being Forged")
plt.ylabel("Density")
plt.title("Predicted Probability Distribution")
plt.legend()
plt.show()


## 14. 5-fold stratified cross-validation
Cross-validation evaluates the model across five different training/validation splits. A pipeline ensures that scaling is fitted separately within each fold, preventing data leakage. This evaluation uses the training portion only; the test set remains separate.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("5-fold cross-validation scores:", np.round(cv_scores, 4))
print("Mean cross-validation accuracy:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))


## 15. Predict the class of a new example
The values below are illustrative. Replace them with feature values from a real banknote record to test another example.

In [ ]:
# Example values: Variance, Skewness, Curtosis, Entropy
new_banknote = pd.DataFrame(
    [[3.6, 8.2, -2.1, -1.0]],
    columns=["Variance", "Skewness", "Curtosis", "Entropy"]
)

new_banknote_scaled = scaler.transform(new_banknote)
prediction = model.predict(new_banknote_scaled)[0]
probabilities = model.predict_proba(new_banknote_scaled)[0]

label = "Genuine" if prediction == 0 else "Forged"

print("Predicted class:", prediction)
print("Predicted label:", label)
print("Probability of Genuine:", round(probabilities[0], 4))
print("Probability of Forged:", round(probabilities[1], 4))


## 16. Conclusion
The Logistic Regression model was trained to classify banknotes as genuine or forged using four numerical features. Its performance was evaluated using accuracy, precision, recall, F1-score, a confusion matrix, ROC-AUC and 5-fold stratified cross-validation. The final results should be interpreted from the outputs generated when this notebook is run.